# 01: Install Python libraries

In [1]:
!pip install ollama

# 02: Update and install packages

In [2]:
!sudo apt update
!usdo apt install -y pciutils
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [107 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.7 MB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [92.7 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,848 kB]
Hit

# 03: Ollama LLMs model tester

In [7]:
from asyncio.unix_events import subprocess
import time
import ollama
import asyncio
import nest_asyncio
import os
import requests

nest_asyncio.apply()


class OllamaModelTester:

    def __init__(self, host, port, models):
        self.host = host
        self.port = port
        self.models = models if models else []
        self.process = None
        self.results = []
        self.dft_sleep_sec = 10
        self.initialization()

    def initialization(self):
        os.environ['OLLAMA_HOST'] = f'{self.host}:{self.port}'
        print('Ollama HOST is initialized')

    def start_serve(self):
        self.process = subprocess.Popen(
            ['ollama', 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ
        )
        time.sleep(self.dft_sleep_sec)
        result = subprocess.run(
            ['curl', '-s', f'http://{self.host}:{self.port}/api/tags'],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print("Ollama serve is started")
        else:
            print("Ollama serve is NOT started")

    def end_serve(self):
        self.process.terminate()
        self.process.wait()
        print("Ollama serve is terminated")

    def pull_model(self, model_name):
        result = subprocess.run(
            ['ollama', 'pull', model_name],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print(f'"{model_name}" model is pulled successfully!')
        else:
            print(f'"{model_name}" model pull threw an error')

    def pull_models(self, models):
        models = models if models else self.models
        for model_name in models:
            result = subprocess.run(
                ['ollama', 'pull', model_name],
                capture_output=True,
                text=True
            )
            if (result.returncode == 0):
                print(f'"{model_name}" model is pulled successfully!')
            else:
                print(f'"{model_name}" model pull threw an error')

    def compare_models(self, prompt_text, models, **options):
        var_models = models if models else self.models
        var_options = {
            'temperature': options.get('temperature', 0.1),
            'num_ctx': options.get('num_ctx', 512)
        }
        for model_name in var_models:
            try:
                print(f'Testing model "{model_name}"')
                start=time.time()

                response = ollama.generate(
                    model=model_name,
                    prompt=prompt_text,
                    options=var_options
                )
                elapsed=time.time() - start
                self.results.append({
                    'model': model_name,
                    'response': response['response'],
                    'tokens': len(response['response'].split()),
                    'elapsed_time': round(elapsed, 2)
                })
                print(f'Successfully completed testing model "{model_name}"')
            except Exception as e:
                self.results.append({
                    'model': model_name,
                    'response': f'"{model_name}" error: {str(e)}',
                    'tokens': 0,
                    'elapsed_time': 0
                })
                print(f'Testing model "{model_name}" threw an error')
        return self.results

    def print_results(self):
        for row in self.results:
            print(f'\nModel:{row['model']}')
            print(f'Tokens:{row['tokens']}')
            print(f'Elapsed Time:{row['elapsed_time']}')
            print(f'Response:{row['response']}\n')

    def __enter__(self):
        self.start_serve()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.end_serve()


if (__name__ == "__main__"):
    i_models = ['tinyllama']
    i_prompt_text = """Summarize this text

Intel Corp. raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning.

The chipmaker priced the offering at $95 per share, according to a company statement. That represents a discount of 6.5% to Friday’s closing price, according to Bloomberg calculations. The share sale drew more than $100 billion in demand, people familiar with the matter said.
"""

    with OllamaModelTester(host='127.0.0.1', port=11434, models=i_models) as om_tester:
        # om_tester.pull_models(models)
        om_tester.compare_models(prompt_text=i_prompt_text, models=None)
        om_tester.print_results()


Ollama HOST is initialized
Ollama serve is started
Testing model "tinyllama"
Successfully completed testing model "tinyllama"

Model:tinyllama
Tokens:47
Elapsed Time:14.76
Response:Intel Corp. Raised $20 billion in an upsized share sale, raising a total of $30 billion, after it initially targeted $25 billion. The chipmaker raised more than $100 billion in demand for its offering, with more than 60% of the shares sold above the initial price range.

Ollama serve is terminated
